# 🎯 FarmTech Solutions - Análise de Clusterização

## Parte 2: Identificação de Padrões e Tendências de Rendimento

**Objetivo:** Aplicar técnicas de Machine Learning não supervisionado para:
1. Identificar grupos/padrões nos dados de rendimento
2. Descobrir tendências de produtividade
3. Detectar cenários discrepantes (outliers)

**Algoritmos utilizados:**
- K-Means Clustering
- DBSCAN (Density-Based Spatial Clustering)
- Hierarchical Clustering (Agglomerative)

---

## 📦 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Pré-processamento
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Clusterização
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

# Métricas
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

# Configurações
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 7)
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('✅ Bibliotecas importadas com sucesso!')

## 📊 2. Carregamento e Preparação dos Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/crop_yield.csv')

print(f'📁 Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas')
print(f'\nPrimeiras linhas:')
df.head()

In [ ]:
# Selecionar features numéricas para clusterização
numeric_features = ['Precipitation (mm day-1)', 'Specific Humidity at 2 Meters (g/kg)', 
                   'Relative Humidity at 2 Meters (%)', 'Temperature at 2 Meters (C)', 'Yield']

X = df[numeric_features].copy()

print('🔢 Features selecionadas para clusterização:')
for i, feat in enumerate(numeric_features, 1):
    print(f'  {i}. {feat}')

print(f'\n📊 Estatísticas descritivas:')
X.describe().T

In [ ]:
# Normalização dos dados (importante para clusterização!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_features)

print('✅ Dados normalizados com StandardScaler')
print(f'\nEstatísticas após normalização (média ≈ 0, std ≈ 1):')
X_scaled_df.describe().T

## 🎯 3. K-Means Clustering

### 3.1 Método do Cotovelo (Elbow Method)

In [ ]:
# Testar diferentes números de clusters
inertias = []
silhouette_scores = []
K_range = range(2, 11)

print('🔄 Testando diferentes números de clusters...')
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f'  K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}')

print('\n✅ Análise concluída!')

In [ ]:
# Visualizar Elbow Method e Silhouette Score
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico do Cotovelo
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Número de Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
ax1.set_title('Método do Cotovelo (Elbow Method)', fontweight='bold', fontsize=14)
ax1.grid(alpha=0.3)
ax1.set_xticks(K_range)

# Gráfico Silhouette Score
ax2.plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Número de Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score por Número de Clusters', fontweight='bold', fontsize=14)
ax2.grid(alpha=0.3)
ax2.set_xticks(K_range)
ax2.axhline(y=max(silhouette_scores), color='g', linestyle='--', alpha=0.5, 
            label=f'Máximo: {max(silhouette_scores):.3f}')
ax2.legend()

plt.tight_layout()
plt.show()

# Identificar o melhor K
best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'\n🎯 Melhor K baseado no Silhouette Score: {best_k}')
print(f'   Silhouette Score: {max(silhouette_scores):.3f}')

### 3.2 Aplicar K-Means com o Melhor K

In [ ]:
# Treinar K-Means com o melhor K
kmeans_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans_best.fit_predict(X_scaled)

print(f'✅ K-Means treinado com K={best_k}')
print(f'\n📊 Distribuição dos clusters:')
print(df['Cluster_KMeans'].value_counts().sort_index())

# Métricas de avaliação
silhouette = silhouette_score(X_scaled, df['Cluster_KMeans'])
davies_bouldin = davies_bouldin_score(X_scaled, df['Cluster_KMeans'])
calinski = calinski_harabasz_score(X_scaled, df['Cluster_KMeans'])

print(f'\n📈 Métricas de Qualidade dos Clusters:')
print(f'  - Silhouette Score: {silhouette:.3f} (quanto maior, melhor)')
print(f'  - Davies-Bouldin Index: {davies_bouldin:.3f} (quanto menor, melhor)')
print(f'  - Calinski-Harabasz Score: {calinski:.2f} (quanto maior, melhor)')

In [ ]:
# Análise dos clusters
print('🔍 Características dos Clusters (K-Means):')
print('='*90)
cluster_analysis = df.groupby('Cluster_KMeans')[numeric_features].mean().round(2)
cluster_analysis

In [ ]:
# Contagem de culturas por cluster
print('\n🌾 Distribuição de Culturas por Cluster:')
print('='*90)
crop_cluster = pd.crosstab(df['Cluster_KMeans'], df['Crop'])
crop_cluster

### 3.3 Visualização dos Clusters (K-Means)

In [ ]:
# PCA para reduzir dimensionalidade e visualizar
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'✅ PCA aplicado - Variância explicada:')
print(f'  - PC1: {pca.explained_variance_ratio_[0]*100:.2f}%')
print(f'  - PC2: {pca.explained_variance_ratio_[1]*100:.2f}%')
print(f'  - Total: {sum(pca.explained_variance_ratio_)*100:.2f}%')

In [ ]:
# Visualização 2D dos clusters
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Clusters coloridos
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster_KMeans'], 
                         cmap='viridis', alpha=0.6, edgecolors='black', linewidth=0.5, s=50)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[0].set_title('K-Means Clustering (PCA)', fontweight='bold', fontsize=14)
axes[0].grid(alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Adicionar centroides
centroids_pca = pca.transform(kmeans_best.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
               marker='X', s=300, c='red', edgecolors='black', linewidth=2, label='Centroides')
axes[0].legend(fontsize=11)

# Plot 2: Yield vs Temperature colorido por cluster
for cluster in range(best_k):
    cluster_data = df[df['Cluster_KMeans'] == cluster]
    axes[1].scatter(cluster_data['Temperature at 2 Meters (C)'], 
                   cluster_data['Yield'], 
                   label=f'Cluster {cluster}', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

axes[1].set_xlabel('Temperature at 2 Meters (C)', fontsize=12)
axes[1].set_ylabel('Yield', fontsize=12)
axes[1].set_title('Clusters: Yield vs Temperature', fontweight='bold', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plot por cluster
from matplotlib import cm

silhouette_vals = silhouette_samples(X_scaled, df['Cluster_KMeans'])

fig, ax = plt.subplots(figsize=(12, 8))
y_lower = 10

for i in range(best_k):
    cluster_silhouette_vals = silhouette_vals[df['Cluster_KMeans'] == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = cm.nipy_spectral(float(i) / best_k)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                     facecolor=color, edgecolor=color, alpha=0.7)
    
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

ax.set_title('Silhouette Plot para K-Means Clustering', fontweight='bold', fontsize=14)
ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.axvline(x=silhouette, color='red', linestyle='--', linewidth=2, label=f'Média: {silhouette:.3f}')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🔍 4. DBSCAN (Density-Based Clustering)

DBSCAN identifica clusters baseados em densidade e detecta outliers automaticamente.

In [ ]:
# Aplicar DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
df['Cluster_DBSCAN'] = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(df['Cluster_DBSCAN'])) - (1 if -1 in df['Cluster_DBSCAN'] else 0)
n_outliers = list(df['Cluster_DBSCAN']).count(-1)

print(f'✅ DBSCAN aplicado')
print(f'\n📊 Resultados:')
print(f'  - Número de clusters encontrados: {n_clusters_dbscan}')
print(f'  - Número de outliers (cluster -1): {n_outliers} ({n_outliers/len(df)*100:.2f}%)')
print(f'\n📈 Distribuição:')
print(df['Cluster_DBSCAN'].value_counts().sort_index())

In [ ]:
# Visualização DBSCAN
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PCA
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster_DBSCAN'], 
                         cmap='Spectral', alpha=0.6, edgecolors='black', linewidth=0.5, s=50)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[0].set_title(f'DBSCAN Clustering - {n_clusters_dbscan} clusters, {n_outliers} outliers', 
                 fontweight='bold', fontsize=14)
axes[0].grid(alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Cluster (-1 = outlier)')

# Plot 2: Yield vs Temperature
unique_clusters = sorted(df['Cluster_DBSCAN'].unique())
for cluster in unique_clusters:
    cluster_data = df[df['Cluster_DBSCAN'] == cluster]
    label = 'Outliers' if cluster == -1 else f'Cluster {cluster}'
    marker = 'x' if cluster == -1 else 'o'
    axes[1].scatter(cluster_data['Temperature at 2 Meters (C)'], 
                   cluster_data['Yield'], 
                   label=label, alpha=0.6, s=50, marker=marker, edgecolors='black', linewidth=0.5)

axes[1].set_xlabel('Temperature at 2 Meters (C)', fontsize=12)
axes[1].set_ylabel('Yield', fontsize=12)
axes[1].set_title('DBSCAN: Yield vs Temperature', fontweight='bold', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🌳 5. Hierarchical Clustering (Agglomerative)

In [ ]:
# Dendrograma (usar amostra para melhor visualização)
sample_size = min(100, len(df))
X_sample = X_scaled[np.random.choice(X_scaled.shape[0], sample_size, replace=False)]

linkage_matrix = linkage(X_sample, method='ward')

plt.figure(figsize=(16, 8))
dendrogram(linkage_matrix)
plt.title(f'Dendrograma - Hierarchical Clustering (amostra de {sample_size})', 
          fontweight='bold', fontsize=14)
plt.xlabel('Índice da Amostra', fontsize=12)
plt.ylabel('Distância', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar Hierarchical Clustering
hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
df['Cluster_Hierarchical'] = hierarchical.fit_predict(X_scaled)

print(f'✅ Hierarchical Clustering aplicado com {best_k} clusters')
print(f'\n📊 Distribuição dos clusters:')
print(df['Cluster_Hierarchical'].value_counts().sort_index())

# Calcular silhouette score
silhouette_hier = silhouette_score(X_scaled, df['Cluster_Hierarchical'])
print(f'\n📈 Silhouette Score: {silhouette_hier:.3f}')

In [ ]:
# Visualização Hierarchical
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PCA
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster_Hierarchical'], 
                         cmap='tab10', alpha=0.6, edgecolors='black', linewidth=0.5, s=50)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[0].set_title('Hierarchical Clustering (PCA)', fontweight='bold', fontsize=14)
axes[0].grid(alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Plot 2: Yield vs Precipitation
for cluster in range(best_k):
    cluster_data = df[df['Cluster_Hierarchical'] == cluster]
    axes[1].scatter(cluster_data['Precipitation (mm day-1)'], 
                   cluster_data['Yield'], 
                   label=f'Cluster {cluster}', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

axes[1].set_xlabel('Precipitation (mm day-1)', fontsize=12)
axes[1].set_ylabel('Yield', fontsize=12)
axes[1].set_title('Hierarchical: Yield vs Precipitation', fontweight='bold', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 6. Comparação dos Métodos de Clusterização

In [ ]:
# Comparação visual dos 3 métodos
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

methods = [
    ('Cluster_KMeans', 'K-Means', 'viridis'),
    ('Cluster_DBSCAN', 'DBSCAN', 'Spectral'),
    ('Cluster_Hierarchical', 'Hierarchical', 'tab10')
]

for idx, (col, title, cmap) in enumerate(methods):
    scatter = axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], c=df[col], 
                               cmap=cmap, alpha=0.6, edgecolors='black', linewidth=0.5, s=40)
    axes[idx].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
    axes[idx].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
    axes[idx].set_title(title, fontweight='bold', fontsize=13)
    axes[idx].grid(alpha=0.3)
    plt.colorbar(scatter, ax=axes[idx], label='Cluster')

plt.suptitle('Comparação dos Métodos de Clusterização (PCA)', 
             fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Tabela comparativa de métricas
print('📊 COMPARAÇÃO DAS MÉTRICAS DE CLUSTERIZAÇÃO')
print('='*90)

# Calcular métricas para todos os métodos (exceto DBSCAN se houver muitos outliers)
metrics_data = []

# K-Means
metrics_data.append({
    'Método': 'K-Means',
    'N° Clusters': best_k,
    'Silhouette': silhouette_score(X_scaled, df['Cluster_KMeans']),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, df['Cluster_KMeans']),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, df['Cluster_KMeans'])
})

# DBSCAN (apenas se tiver clusters válidos)
if n_clusters_dbscan > 1:
    # Filtrar outliers para cálculo de métricas
    df_no_outliers = df[df['Cluster_DBSCAN'] != -1]
    X_no_outliers = X_scaled[df['Cluster_DBSCAN'] != -1]
    
    metrics_data.append({
        'Método': 'DBSCAN',
        'N° Clusters': n_clusters_dbscan,
        'Silhouette': silhouette_score(X_no_outliers, df_no_outliers['Cluster_DBSCAN']),
        'Davies-Bouldin': davies_bouldin_score(X_no_outliers, df_no_outliers['Cluster_DBSCAN']),
        'Calinski-Harabasz': calinski_harabasz_score(X_no_outliers, df_no_outliers['Cluster_DBSCAN'])
    })

# Hierarchical
metrics_data.append({
    'Método': 'Hierarchical',
    'N° Clusters': best_k,
    'Silhouette': silhouette_score(X_scaled, df['Cluster_Hierarchical']),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, df['Cluster_Hierarchical']),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, df['Cluster_Hierarchical'])
})

comparison_df = pd.DataFrame(metrics_data)
comparison_df = comparison_df.round(3)
print(comparison_df.to_string(index=False))

print('\n💡 Interpretação:')
print('  - Silhouette: Quanto MAIOR, melhor (valores entre -1 e 1)')
print('  - Davies-Bouldin: Quanto MENOR, melhor')
print('  - Calinski-Harabasz: Quanto MAIOR, melhor')

## 📝 7. Análise e Interpretação dos Clusters

In [ ]:
# Análise detalhada dos clusters K-Means (melhor método)
print('🔍 ANÁLISE DETALHADA DOS CLUSTERS (K-Means)')
print('='*90)

for cluster_id in range(best_k):
    cluster_data = df[df['Cluster_KMeans'] == cluster_id]
    
    print(f'\n📌 CLUSTER {cluster_id}:')
    print(f'  - Tamanho: {len(cluster_data)} registros ({len(cluster_data)/len(df)*100:.1f}%)')
    print(f'  - Yield médio: {cluster_data["Yield"].mean():.2f}')
    print(f'  - Temperatura média: {cluster_data["Temperature at 2 Meters (C)"].mean():.2f}°C')
    print(f'  - Precipitação média: {cluster_data["Precipitation (mm day-1)"].mean():.2f} mm/dia')
    print(f'  - Culturas dominantes: {cluster_data["Crop"].value_counts().head(3).to_dict()}')

print('\n' + '='*90)

In [ ]:
# Boxplot de Yield por cluster
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# K-Means
df.boxplot(column='Yield', by='Cluster_KMeans', ax=axes[0], patch_artist=True)
axes[0].set_title('Yield por Cluster (K-Means)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Cluster', fontsize=11)
axes[0].set_ylabel('Yield', fontsize=11)

# DBSCAN
df.boxplot(column='Yield', by='Cluster_DBSCAN', ax=axes[1], patch_artist=True)
axes[1].set_title('Yield por Cluster (DBSCAN)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Cluster (-1 = outlier)', fontsize=11)
axes[1].set_ylabel('Yield', fontsize=11)

# Hierarchical
df.boxplot(column='Yield', by='Cluster_Hierarchical', ax=axes[2], patch_artist=True)
axes[2].set_title('Yield por Cluster (Hierarchical)', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Cluster', fontsize=11)
axes[2].set_ylabel('Yield', fontsize=11)

plt.suptitle('')  # Remover título automático
plt.tight_layout()
plt.show()

## 📝 8. Conclusões da Clusterização

In [ ]:
print('📊 CONCLUSÕES DA ANÁLISE DE CLUSTERIZAÇÃO')
print('='*90)
print(f'\n1. NÚMERO ÓTIMO DE CLUSTERS: {best_k}')
print(f'   Baseado no Silhouette Score máximo de {max(silhouette_scores):.3f}')

print(f'\n2. MELHOR MÉTODO:')
best_method = comparison_df.loc[comparison_df['Silhouette'].idxmax(), 'Método']
print(f'   {best_method} apresentou o melhor Silhouette Score')

print(f'\n3. PADRÕES IDENTIFICADOS:')
print(f'   - Os clusters mostram diferenças significativas no rendimento')
print(f'   - Condições climáticas (temperatura, precipitação) influenciam a formação dos grupos')
print(f'   - DBSCAN identificou {n_outliers} outliers ({n_outliers/len(df)*100:.2f}% dos dados)')

print(f'\n4. TENDÊNCIAS DE PRODUTIVIDADE:')
for i in range(best_k):
    cluster_yield = df[df['Cluster_KMeans'] == i]['Yield'].mean()
    trend = 'ALTA' if cluster_yield > df['Yield'].mean() else 'BAIXA'
    print(f'   - Cluster {i}: Produtividade {trend} (Yield médio: {cluster_yield:.2f})')

print('\n' + '='*90)
print('🎯 PRÓXIMO PASSO: Desenvolvimento de Modelos Preditivos de Regressão')
print('='*90)

In [ ]:
# Salvar DataFrame com clusters para uso futuro
df.to_csv('crop_yield_com_clusters.csv', index=False)
print('✅ DataFrame com clusters salvo em: crop_yield_com_clusters.csv')